# BiLSTM Training for Distraction Prediction

Train the baseline BiLSTM with temporal attention and save the best validation checkpoint.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        marker = candidate / "distraction_prediction" / "models" / "lstm_model.py"
        if marker.exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from distraction_prediction.models.lstm_model import DistractionLSTM

WINDOWS_DIR = PROJECT_ROOT / "distraction_prediction" / "data" / "processed" / "windows"
SAVE_DIR = PROJECT_ROOT / "distraction_prediction" / "models" / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
X_train = np.load(WINDOWS_DIR / "X_train.npy")
y_train = np.load(WINDOWS_DIR / "y_train.npy")
X_val = np.load(WINDOWS_DIR / "X_val.npy")
y_val = np.load(WINDOWS_DIR / "y_val.npy")

print(f"Train: {X_train.shape}, distracted rate={y_train.mean():.3f}")
print(f"Val  : {X_val.shape}, distracted rate={y_val.mean():.3f}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_items = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        total_correct += (predictions == labels).sum().item()
        total_items += len(labels)

    return total_loss / total_items, total_correct / total_items


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_items = 0.0, 0, 0
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (predictions == labels).sum().item()
            total_items += len(labels)

    return total_loss / total_items, total_correct / total_items

In [ ]:
batch_size = 256
max_epochs = 100
early_stopping_patience = 15
learning_rate = 5e-4
weight_decay = 1e-3

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train)),
    batch_size=batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val)),
    batch_size=batch_size,
)

model = DistractionLSTM(input_size=X_train.shape[2], head_name="out").to(device)
pos_weight = torch.tensor([(y_train == 0).sum() / y_train.sum()], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
best_val_loss = float("inf")
epochs_without_improvement = 0
model_path = SAVE_DIR / "best_model.pt"
history = []
start_time = time.time()

for epoch in range(1, max_epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate_epoch(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
    })

    message = (
        f"Epoch {epoch:03d} | "
        f"Train {train_loss:.4f}/{train_acc:.4f} | "
        f"Val {val_loss:.4f}/{val_acc:.4f} | "
        f"LR {optimizer.param_groups[0]['lr']:.1e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_loss": val_loss,
                "config": {
                    "input_size": X_train.shape[2],
                    "hidden_size": 64,
                    "num_layers": 1,
                    "dropout": 0.5,
                    "bidirectional": True,
                    "attention_activation": "tanh",
                    "head_name": "out",
                },
            },
            model_path,
        )
        message += "  [saved]"
    else:
        epochs_without_improvement += 1

    print(message)

    if epochs_without_improvement >= early_stopping_patience:
        print(f"Early stopping at epoch {epoch}")
        break

elapsed = time.time() - start_time
print(f"\nTraining finished in {elapsed:.1f}s")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Saved model: {model_path}")